# Synthetic Data Generation for RAG Evaluation

Session 1 built a vector RAG application over a cat health guideline PDF. This
session creates an evaluation dataset for that application and uses the dataset
to compare two retrieval configurations. All generation, embedding, RAG, and
judge requests are routed through Vercel AI Gateway.

The workflow is:

~~~text
corpus -> knowledge graph -> synthetic examples -> human review
       -> LangSmith dataset -> baseline and candidate experiments
~~~

Synthetic examples reduce the cost of getting started, but generated references
are not automatically ground truth. We will inspect and curate them before using
them as evaluation targets.

> This is an educational cat health exercise, not veterinary advice. Generated
> questions and answers must not be used to diagnose, prescribe, or replace a
> veterinarian.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how Ragas builds a knowledge graph for test data generation.
- Distinguish single-hop specific, multi-hop specific, and multi-hop abstract queries.
- Generate and review synthetic questions, reference contexts, and reference answers.
- Route generation, embeddings, RAG, and judge calls through Vercel AI Gateway.
- Upload reviewed examples to a LangSmith dataset.
- Evaluate answer correctness, answer groundedness, and retrieval relevance.
- Run a controlled RAG experiment that changes one variable at a time.

## Table of Contents

- **Breakout Room #1: Synthetic Test Data with Ragas**
  - Task 1: Environment Setup
  - Task 2: Load the Cat Health Corpus
  - Task 3: Build and Enrich a Knowledge Graph
  - Task 4: Inspect the Query Distribution
  - Task 5: Generate and Inspect a Synthetic Test Set
  - Activity #1: Review and Curate the Dataset
- **Breakout Room #2: RAG Evaluation with LangSmith**
  - Task 6: Create a LangSmith Dataset
  - Task 7: Build a Baseline RAG Application
  - Task 8: Define RAG Evaluators
  - Task 9: Run the Baseline Experiment
  - Task 10: Change One Retrieval Variable and Re-Evaluate
  - Activity #2: Compare, Diagnose, and Iterate
  - Advanced Build: Add Robustness and Adversarial Cases

---
# Breakout Room #1
## Synthetic Test Data with Ragas

Ragas uses the source corpus to create a richer representation of its topics and
relationships. Query synthesizers then select scenarios from that representation
and generate questions plus reference answers.

The knowledge graph is a generation aid. It is not the graph used by the RAG
application in Breakout Room #2.

## Task 1: Environment Setup

From the <code>05_Synthetic_Data_Generation_for_RAG_Evals</code> folder:

~~~bash
uv sync
~~~

Then select the environment created by uv as this notebook's kernel.

Required accounts:

- Vercel AI Gateway for generation, embeddings, the RAG answer model, and judges
- LangSmith for the dataset and experiments

A direct OpenAI API key is not required. The OpenAI SDK is used only as a
protocol-compatible client pointed at Vercel AI Gateway.

The default synthetic test set is intentionally small. Ragas generation and
LLM-as-judge evaluation both make multiple model calls, so start small and scale
only after inspecting quality.

### Imports

In [1]:
from __future__ import annotations

import os
from collections import Counter
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import instructor
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, field_validator

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client, evaluate
from openevals.llm import create_llm_as_judge
from openevals.prompts import (
    CORRECTNESS_PROMPT,
    RAG_GROUNDEDNESS_PROMPT,
    RAG_RETRIEVAL_RELEVANCE_PROMPT,
)

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.synthesizers import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    SingleHopSpecificQuerySynthesizer,
    default_query_distribution,
)
from ragas.testset.transforms import (
    CustomNodeFilter,
    SummaryExtractor,
    apply_transforms,
    default_transforms_for_prechunked,
)

/Users/aneeta/Desktop/practice/The-AI-Engineering-Certification-v1.0/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### API Keys, Models, and Cost Controls

The notebook reads model names and budgets from environment variables so you can
tune cost without editing every cell. Vercel AI Gateway exposes an
OpenAI-compatible endpoint, so the OpenAI and LangChain clients only need a
different API key, base URL, and provider-qualified model ID.

See the [Vercel AI Gateway Python documentation](https://vercel.com/docs/ai-gateway/sdks-and-apis/python)
for the current authentication and endpoint details.

LangSmith uses <code>LANGSMITH_TRACING</code>. The older
<code>LANGCHAIN_TRACING_V2</code> name from the source notebook is no longer
needed here.

In [2]:
gateway_api_key = (
    os.environ.get("AI_GATEWAY_API_KEY")
    or os.environ.get("VERCEL_OIDC_TOKEN")
)

if not gateway_api_key:
    gateway_api_key = getpass("Vercel AI Gateway API Key: ")
    os.environ["AI_GATEWAY_API_KEY"] = gateway_api_key

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API Key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault(
    "LANGSMITH_PROJECT",
    "aim-session-5-synthetic-rag-evals",
)

GATEWAY_BASE_URL = os.environ.get(
    "AI_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)
GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)
RAG_MODEL_NAME = os.environ.get(
    "AIM_RAG_MODEL",
    "openai/gpt-5.4-mini",
)
JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)
EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)
TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "6"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "2"))

gateway_models = {
    "generator": GENERATOR_MODEL_NAME,
    "rag": RAG_MODEL_NAME,
    "judge": JUDGE_MODEL_NAME,
    "embedding": EMBEDDING_MODEL_NAME,
}
for role, model_name in gateway_models.items():
    if "/" not in model_name:
        raise ValueError(
            f"{role} model must use a provider-qualified AI Gateway ID: "
            f"{model_name!r}"
        )

print(f"Ragas: {version('ragas')}")
print(f"LangSmith: {version('langsmith')}")
print(f"AI Gateway: {GATEWAY_BASE_URL}")
print(f"Generator model: {GENERATOR_MODEL_NAME}")
print(f"RAG model: {RAG_MODEL_NAME}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Synthetic examples: {TESTSET_SIZE}")
print(f"LangSmith tracing: {os.environ['LANGSMITH_TRACING']}")

Ragas: 0.4.4.dev8+g298b68274
LangSmith: 0.8.18
AI Gateway: https://ai-gateway.vercel.sh/v1
Generator model: openai/gpt-5.4-mini
RAG model: openai/gpt-5.4-mini
Judge model: openai/gpt-5.4-mini
Embedding model: openai/text-embedding-3-small
Synthetic examples: 6
LangSmith tracing: true


## Task 2: Load the Cat Health Corpus

The corpus is the bundled 2021 AAHA/AAFP Feline Life Stage Guidelines PDF used
in Session 1. <code>PyPDFLoader</code> extracts one LangChain document per page,
including page metadata that survives later chunking.

This gives the generator multiple related units to connect:

- hydration and urinary signs
- preventive care and senior care
- dental pain and behavior changes
- monitoring and emergency escalation

In [3]:
corpus_path = Path("data/cat_health_guidelines.pdf")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

pdf_loader = PyPDFLoader(str(corpus_path))
source_documents = pdf_loader.load()
source_documents = [
    document
    for document in source_documents
    if len(document.page_content.strip()) >= 200
]

for index, document in enumerate(source_documents):
    page_number = int(document.metadata.get("page", index)) + 1
    document.metadata.update(
        {
            "source": corpus_path.name,
            "document_type": "feline_life_stage_guidelines",
            "page_number": page_number,
        }
    )

print(f"Loaded {len(source_documents)} text-containing PDF pages")
for document in source_documents[:5]:
    page_number = document.metadata["page_number"]
    print(
        f"- page {page_number}: "
        f"{len(document.page_content)} characters"
    )

Loaded 20 text-containing PDF pages
- page 1: 4913 characters
- page 2: 2084 characters
- page 3: 5955 characters
- page 6: 5673 characters
- page 7: 3588 characters


Inspect one PDF page and its metadata. The metadata is useful for debugging,
trace inspection, and explaining where a retrieved chunk came from.

In [5]:
sample_document = source_documents[19]

print(sample_document.metadata)
print()
print(sample_document.page_content[:800])

{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 21, 'page_label': '22', 'document_type': 'feline_life_stage_guidelines', 'page_number': 22}

130. van Bree FPJ, Bokken GCAM, Mineur R, et al. Zoonotic bacteria and
parasites found in raw meat-based diets for cats and dogs. V et Rec2018;
182:50. DOI: 10.1136/vr.104535.
131. Carney HC, W ard CR, Bailey SJ, et al. 2016 AAFP guidelines for the
management of feline hyperthyroidism.JF e l i n eM e dS u r g2016;18:400–16.
132. Sparkes AH, Caney S, Chalhoub S, et al. ISFM consensus guidelines on
the diagnosis and management of feline chronic kidney disease. J Feline
Med Surg 2016;18:219–39.
133. Behrend E, Holford A, Lathan P , et al. 2018 AAHA diabetes manage-
ment guidelines for dogs and cats. J 

## Task 3: Build and Enrich a Knowledge Graph

The unrolled workflow makes the generation stages visible:

1. Treat each text-containing PDF page as a pre-chunked Ragas node.
2. Run Ragas extractors, embeddings, and relationship builders.
3. Save the graph so expensive enrichment can be reused.

Ragas remains responsible for graph enrichment and synthetic generation. The
newer pinned Ragas build exposes an official Instructor mode parameter, so its
LLM factory can use AI Gateway tool calls directly without custom wrappers.

In [6]:
gateway_client = OpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=4096,
)
# Provider-qualified Gateway IDs bypass Ragas's GPT-5 parameter detection.
# Keep only the token limit supported by the Gateway route. max_retries is
# consumed locally by Instructor and is not sent to AI Gateway.
generator_llm.model_args = {
    "max_tokens": 4096,
    "max_retries": 3,
}

generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_client,
)

ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

/var/folders/f7/6j1mkm0j4dx3y7g7y5lvjd600000gn/T/ipykernel_51221/1199448542.py:21: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = embedding_factory(


Before building the graph, make one small structured-output request through
Ragas. This catches gateway authentication, model availability, and tool-calling
incompatibilities without waiting for every PDF page to retry.

In [8]:
class GatewayToolCallCheck(BaseModel):
    status: str


class NonEmptySummary(BaseModel):
    text: str

    @field_validator("text")
    @classmethod
    def require_text(cls, value):
        value = value.strip()
        if not value:
            raise ValueError("summary text cannot be empty")
        return value


gateway_check = generator_llm.generate(
    "Use the required tool with a short, non-empty status message.",
    GatewayToolCallCheck,
)
if not gateway_check.status.strip():
    raise RuntimeError("AI Gateway returned an empty tool-call check")

print(f"AI Gateway tool-based structured output: {gateway_check.status}")

AI Gateway tool-based structured output: ok


In [9]:
def build_prechunked_knowledge_graph(chunks):
    return KnowledgeGraph(
        nodes=[
            Node(
                type=NodeType.CHUNK,
                properties={
                    "page_content": chunk.page_content,
                    "document_metadata": dict(chunk.metadata),
                },
            )
            for chunk in chunks
            if chunk.page_content.strip()
        ]
    )


generation_chunks = list(source_documents)
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)

print(f"Ragas input chunks: {len(generation_chunks)}")
print(knowledge_graph)

Ragas input chunks: 20
KnowledgeGraph(nodes: 20, relationships: 0)


### Apply Ragas Transforms

Because the PDF loader already gives us coherent page-level chunks, use Ragas'
built-in pre-chunked transform pipeline. It skips headline extraction and
splitting, then applies Ragas summaries, embeddings, themes, named entities,
and relationship builders directly to the PDF pages. The parent-child node
filter is omitted because these page chunks intentionally have no parent nodes.
A non-empty output constraint makes Instructor retry blank Ragas summaries before
the built-in embedding transform runs.

In [12]:
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)
transforms = [
    transform
    for transform in default_transforms_for_prechunked(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )
    if not isinstance(transform, CustomNodeFilter)
]

summary_transform = next(
    transform
    for transform in transforms
    if isinstance(transform, SummaryExtractor)
)
summary_transform.prompt.output_model = NonEmptySummary

print("Ragas transform pipeline:")
for transform in transforms:
    nested = getattr(transform, "transformations", None)
    if nested is None:
        print(f"- {type(transform).__name__}")
    else:
        names = ", ".join(type(item).__name__ for item in nested)
        print(f"- Parallel({names})")

for transform in transforms:
    apply_transforms(
        knowledge_graph,
        transform,
        run_config=ragas_run_config,
    )
    if isinstance(transform, SummaryExtractor):
        empty_summary_nodes = [
            node
            for node in knowledge_graph.nodes
            if not str(node.get_property("summary") or "").strip()
        ]
        if empty_summary_nodes:
            raise RuntimeError(
                "Ragas did not produce non-empty summaries for "
                f"{len(empty_summary_nodes)} PDF chunks"
            )

print(knowledge_graph)

Ragas transform pipeline:
- SummaryExtractor
- Parallel(EmbeddingExtractor, ThemesExtractor, NERExtractor)
- Parallel(CosineSimilarityBuilder, OverlapScoreBuilder)


Applying SummaryExtractor: 100%|██████████| 20/20 [00:44<00:00,  2.25s/it]
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/60 [00:00<?, ?it/s]/Users/aneeta/Desktop/practice/The-AI-Engineering-Certification-v1.0/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/ragas/testset/transforms/base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]: 100%|██████████| 60/60 [01:03<00:00,  1.06s/it]
Applying [CosineSimilarityBuilder, OverlapScoreBuilder]: 100%|██████████| 2/2 [00:00<00:00, 107.23it/s]

KnowledgeGraph(nodes: 20, relationships: 52)


Inspect the graph at a high level. Different Ragas versions may add different
properties, so the notebook avoids depending on one exact internal schema.

In [13]:
node_type_counts = Counter(str(node.type) for node in knowledge_graph.nodes)

print("Node types:")
for node_type, count in node_type_counts.items():
    print(f"- {node_type}: {count}")

print(f"Relationships: {len(knowledge_graph.relationships)}")

for index, node in enumerate(knowledge_graph.nodes[:3], start=1):
    property_names = sorted(node.properties.keys())
    print(f"Node {index} properties: {property_names}")

Node types:
- NodeType.CHUNK: 20
Relationships: 52
Node 1 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 2 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 3 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']


### Save and Reload the Graph

Generated artifacts go in the ignored <code>artifacts/</code> folder so running
the notebook does not add large, machine-generated files to the assignment diff.

In [14]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

knowledge_graph_path = artifacts_dir / "cat_health_knowledge_graph.json"
knowledge_graph.save(str(knowledge_graph_path))

loaded_knowledge_graph = KnowledgeGraph.load(str(knowledge_graph_path))

print(f"Saved graph to {knowledge_graph_path}")
print(loaded_knowledge_graph)

Saved graph to artifacts/cat_health_knowledge_graph.json
KnowledgeGraph(nodes: 20, relationships: 52)


In [15]:
relationship_types = Counter(r.type for r in knowledge_graph.relationships)
print(relationship_types)

Counter({'summary_similarity': 40, 'entities_overlap': 12})


#### ❓ Question #1

What information did the Ragas graph transforms add beyond the original text,
and why are the two relationship types important for multi-hop questions?

##### ✅ Answer
The transforms enriched each page node with summaries, extracted entities and themes, and embeddings of both the page content and the summary, then used those properties to draw edges between related pages. My graph has two relationship types: summary_similarity (40 edges, thematic relatedness between page summaries) and entities_overlap (12 edges, shared concrete concepts between pages). Both matter for multi-hop questions because the synthesizers traverse these edges to find chunk pairs worth combining: summary_similarity enables abstract multi-hop questions across themes, and entities_overlap enables specific multi-hop questions that combine concrete facts.



## Task 4: Inspect the Query Distribution

Ragas can synthesize several kinds of questions:

| Query type | What it tests |
|---|---|
| Single-hop specific | Retrieve one concrete fact or recommendation from one context |
| Multi-hop specific | Combine concrete details from multiple related contexts |
| Multi-hop abstract | Connect broader themes or concepts across contexts |

The distribution is part of the evaluation specification. It determines which
behaviors are common in the generated dataset.

In [16]:
query_distribution = default_query_distribution(
    generator_llm,
    kg=loaded_knowledge_graph,
)

print("Available query synthesizers:")
for synthesizer, weight in query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

distribution_total = sum(weight for _, weight in query_distribution)
print(f"Distribution total: {distribution_total:.2f}")

Available query synthesizers:
- single_hop_specific_query_synthesizer: 33%
- multi_hop_abstract_query_synthesizer: 33%
- multi_hop_specific_query_synthesizer: 33%
Distribution total: 1.00


### Define a Custom Distribution

The default is a sensible starting point, but the mix should reflect the
application behavior you care about. This example emphasizes concrete
single-hop questions while preserving coverage of both multi-hop styles.

Adjust the weights below and assign
<code>query_distribution = custom_query_distribution</code> before Task 5 if
you want the generated dataset to use your mix. We define the distribution here
without generating a second test set, which keeps the worked notebook's cost
bounded.

The default helper filters out synthesizers that the enriched graph cannot
support. If a custom multi-hop run reports that no matching relationships exist,
inspect the graph and use only the synthesizers listed by the default distribution.

In [17]:
custom_query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(llm=generator_llm),
        0.50,
    ),
    (
        MultiHopSpecificQuerySynthesizer(llm=generator_llm),
        0.30,
    ),
    (
        MultiHopAbstractQuerySynthesizer(llm=generator_llm),
        0.20,
    ),
]

assert abs(
    sum(weight for _, weight in custom_query_distribution) - 1.0
) < 1e-9

for synthesizer, weight in custom_query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

- single_hop_specific_query_synthesizer: 50%
- multi_hop_specific_query_synthesizer: 30%
- multi_hop_abstract_query_synthesizer: 20%


In [20]:
query_distribution = default_query_distribution(
    generator_llm,
    kg=loaded_knowledge_graph,
)

print("Available query synthesizers:")
for synthesizer, weight in query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

Available query synthesizers:
- single_hop_specific_query_synthesizer: 33%
- multi_hop_abstract_query_synthesizer: 33%
- multi_hop_specific_query_synthesizer: 33%


#### ❓ Question #2

Describe the three query types in your own words. Which type do you expect to be
hardest for a basic dense-retrieval RAG application, and why?

##### ✅ Answer

Single_hop is the simplest, since the query returns a single retrieval. Multi_hop_specific requires information from multiple sources, which requires the system to make information from different parts of the knowledge base. 

Multi_hop_abstract is most complex; system has to synthesize information from multiple sources and the response is not said word for word in knowledge base. 

Multi_hop_abstract to be the hardest for a basic dense-retrieval RAG. Dense retrieval ranks individual chunks by embedding similarity to the question, but abstract questions are phrased at a higher conceptual level than any individual chunk contains. The retriever pulls semantically nearby but conceptually thin chunks, and the model has to assemble a thematic answer from fragments that don't individually contain it.




## Task 5: Generate and Inspect a Synthetic Test Set

Each generated row contains:

- <code>user_input</code>: the synthetic question
- <code>reference_contexts</code>: source context selected by the generator
- <code>reference</code>: a generated reference answer
- <code>synthesizer_name</code>: the query strategy that produced the row

The reference is generated from selected source context. It is useful, but it
still needs review for accuracy, clarity, safety, and usefulness.

In [19]:
testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=loaded_knowledge_graph,
)

synthetic_testset = testset_generator.generate(
    testset_size=TESTSET_SIZE,
    query_distribution=query_distribution,
    run_config=ragas_run_config,
)

testset_df = synthetic_testset.to_pandas()

display(
    testset_df[
        [
            "user_input",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Generating Samples: 100%|██████████| 6/6 [00:19<00:00,  3.24s/it]


,user_input,reference,synthesizer_name
0,Who is Theresa DePorter in the feline life sta...,Theresa DePorter is listed among the authors o...,single_hop_specific_query_synthesizer
1,How do the 2021 AAHA/AAFP Feline Life Stage Gu...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,single_hop_specific_query_synthesizer
2,How do reduced-stress handling and feline-frie...,The context explains that taking time to evalu...,multi_hop_abstract_query_synthesizer
3,How can pheromones and behavioral modification...,"Cats may be distressed while hiding anxiety, s...",multi_hop_abstract_query_synthesizer
4,How are osteoarthritis-related senior cat care...,The context links osteoarthritis and senior-ca...,multi_hop_specific_query_synthesizer
5,"In the United States, how do the 2021 AAHA/AAF...",The context says that the feline patient’s lif...,multi_hop_specific_query_synthesizer


In [14]:
testset_path = artifacts_dir / "cat_health_synthetic_testset.jsonl"
testset_df.to_json(
    testset_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Examples by synthesizer:")
print(testset_df["synthesizer_name"].value_counts())
print()
print(f"Saved candidate examples to {testset_path}")

Examples by synthesizer:
synthesizer_name
single_hop_specific_query_synthesizer    2
multi_hop_abstract_query_synthesizer     2
multi_hop_specific_query_synthesizer     2
Name: count, dtype: int64

Saved candidate examples to artifacts/cat_health_synthetic_testset.jsonl


### Abstracted Ragas Alternative

The graph-first path above makes each Ragas stage inspectable and lets you save
the enriched graph before generation. Ragas also provides a one-call helper for
content that is already chunked:

~~~python
quick_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
quick_testset = quick_generator.generate_with_chunks(
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=transforms,
    run_config=ragas_run_config,
)
~~~

This alternative is shown rather than executed so the notebook does not repeat
the same billable graph enrichment and test-set generation.

#### ❓ Question #3

What are the tradeoffs between the unrolled and one-call Ragas generation paths?
When would you choose each one?

##### ✅ Answer

Unrolled is easier to debug, but harder to maintain in production. There is more control, so sort of like the Linear Regression of ML
One-call is harder to debug, but easier to maintain with less code. There's less control and harder to debug. Thing XGBoost. 

Unrolled = better for debugging and control.Think POC.  One-call when you want to write less code to maintain and easier in Production. 


## 🏗️ Activity #1: Review and Curate the Dataset

Review every generated row before uploading it.

For each example, check:

1. Is the question answerable from the reference contexts?
2. Is the reference answer fully supported by those contexts?
3. Is the wording natural for a plausible user?
4. Does the example duplicate another row?
5. Does it preserve the corpus's medical-safety boundaries?

Requirements:

- Remove or repair at least one weak example, if one exists.
- Record why you kept, edited, or removed it.
- Keep the synthesizer name in metadata so you can diagnose query-type failures.

In [21]:
for i, row in testset_df.iterrows():
    print(f"--- Row {i} ({row['synthesizer_name']}) ---")
    print(f"Q: {row['user_input']}")
    print(f"A: {row['reference'][:400]}")
    print()

--- Row 0 (single_hop_specific_query_synthesizer) ---
Q: Who is Theresa DePorter in the feline life stage guidelines?
A: Theresa DePorter is listed among the authors of the 2021 AAHA/AAFP Feline Life Stage Guidelines as "Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM."

--- Row 1 (single_hop_specific_query_synthesizer) ---
Q: How do the 2021 AAHA/AAFP Feline Life Stage Guidelines help veterinarians assess and manage a cat's healthcare across its lifetime?
A: The 2021 AAHA/AAFP Feline Life Stage Guidelines provide a comprehensive age-associated framework for promoting health and longevity throughout a cat's lifetime. They were developed by a task force of experts in feline clinical medicine, and their recommendations are intended to guide individualized risk assessment, preventive healthcare strategies, and treatment pathways that evolve as the cat mat

--- Row 2 (multi_hop_abstract_query_synthesizer) ---
Q: How do reduced-stress handling and feline-friendly handling support an individual

In [22]:
# Activity #1 workspace
#
# Review decisions:
#   Row 0 — REMOVED: authorship trivia ("Who is Theresa DePorter..."). Not a 
#     realistic cat health query; tests PDF metadata, not health capability.
#   Row 3 — REPAIRED: Ragas typo'd the question ("distreeess and anxiiety").
#     Fixed spelling, kept original intent.
#   Rows 1, 2, 4, 5 — KEPT: answerable from reference contexts, references 
#     grounded in corpus, natural wording, no duplicates, no safety violations.

approved_testset_df = testset_df.drop(index=[0]).reset_index(drop=True)

# After drop+reset, original Row 3 is now at index 2
approved_testset_df.loc[2, "user_input"] = (
    "How can pheromones and behavioral modification help with distress and "
    "anxiety in cats, and what signs show a cat is stressed?"
)

review_status = "student_reviewed"

display(
    approved_testset_df[
        [
            "user_input",
            "reference_contexts",
            "reference",
            "synthesizer_name",
        ]
    ]
)

,user_input,reference_contexts,reference,synthesizer_name
0,How do the 2021 AAHA/AAFP Feline Life Stage Gu...,[Introduction\nThe feline patient ’s life stag...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,single_hop_specific_query_synthesizer
1,How do reduced-stress handling and feline-frie...,"[<1-hop>\n\nFor example, some senior cats aged...",The context explains that taking time to evalu...,multi_hop_abstract_query_synthesizer
2,How can pheromones and behavioral modification...,[<1-hop>\n\nDetecting signs of pain or anxiety...,"Cats may be distressed while hiding anxiety, s...",multi_hop_abstract_query_synthesizer
3,How are osteoarthritis-related senior cat care...,[<1-hop>\n\nchallenges of diagnosing feline ar...,The context links osteoarthritis and senior-ca...,multi_hop_specific_query_synthesizer
4,"In the United States, how do the 2021 AAHA/AAF...","[<1-hop>\n\ndisease, masses, or orofacial pain...",The context says that the feline patient’s lif...,multi_hop_specific_query_synthesizer


### 📝 Activity #1 Notes


- **Row 0 removed** — "Who is Theresa DePorter in the feline life stage guidelines?" This is authorship trivia drawn from the document header. No cat owner or vet using a cat health RAG would ask who one of the document's authors is. It tests PDF metadata retrieval, not health knowledge. The reference answer is technically supported by the contexts (the author list is there), but the question fails the plausible-user criterion. No safety issue, just out of scope for the app's actual use case.

- **Row 3 repaired** — Original question: "How can pheromones and behavioral modification help with distreeess and anxiiety in cats..." Ragas generated typos in two words ("distreeess", "anxiiety"). The underlying question about pheromones, behavioral modification, and stress signs is legitimate cat health content. The reference answer is well-grounded in the contexts (covers stress signs like flattened ears, dilated pupils, hiding, and the role of synthetic facial pheromones). Only the typos were fixed; the question intent was preserved.

- **Rows 1, 2, 4, 5 kept** — All four pass every check:
  - Answerable from reference contexts (multi-hop rows have <1-hop> markers showing which chunk anchors each part of the answer)
  - Reference answers grounded in the PDF content (life-stage framework, reduced-stress handling, osteoarthritis-related care, vaccine/dentistry guidance)
  - Wording reads as a plausible question from a vet professional or informed cat owner. Some phrasing is formal (e.g. references to "the 2021 AAHA/AAFP Feline Life Stage Guidelines"), but that's consistent with how a vet or clinic manager might actually phrase a question against this corpus
  - No duplicates across rows
  - No medical-safety boundary violations: no diagnostic requests, no medication doses, no treatment prescriptions

- **Synthesizer distribution preserved** — After curation, the 5 retained rows still cover all three synthesizer types (1 single-hop specific, 2 multi-hop abstract, 2 multi-hop specific). This matters because the `synthesizer_name` metadata is preserved through the LangSmith upload, which lets me diagnose per-query-type failure patterns later when comparing experiments.

- **Safety review** — No retained example asked for a diagnosis, medication dose, or treatment prescription. All five stay within the corpus's educational scope.

---
# Breakout Room #2
## RAG Evaluation with LangSmith

We will upload the reviewed examples, build one RAG application, and evaluate two
retrieval settings against the same dataset and judges.

Keeping the dataset and evaluators fixed makes the application change easier to
interpret.

## Task 6: Create a LangSmith Dataset

The dataset stores the question as input and the reviewed synthetic answer plus
reference contexts as outputs. Query type and provenance remain metadata.

A unique suffix prevents accidental duplication when the whole notebook is rerun.
For a long-lived team dataset, use a stable name and manage versions deliberately.

In [23]:
def as_string_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if hasattr(value, "tolist"):
        converted = value.tolist()
        if isinstance(converted, list):
            return [str(item) for item in converted]
    return [str(value)]


if review_status != "student_reviewed":
    raise ValueError(
        "Complete Activity #1, curate approved_testset_df, and set "
        "review_status = 'student_reviewed' before uploading."
    )


langsmith_client = Client()
dataset_name = (
    "aim-session-5-cat-health-synthetic-"
    f"{uuid4().hex[:8]}"
)

langsmith_dataset = langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description=(
        "Ragas-generated questions for the AI Makerspace "
        "cat health RAG lesson."
    ),
    metadata={
        "session": 5,
        "source": "ragas",
        "corpus": str(corpus_path),
    },
)

langsmith_examples = []
for _, row in approved_testset_df.iterrows():
    langsmith_examples.append(
        {
            "inputs": {
                "question": str(row["user_input"]),
            },
            "outputs": {
                "answer": str(row["reference"]),
                "reference_contexts": as_string_list(
                    row["reference_contexts"]
                ),
            },
            "metadata": {
                "synthesizer_name": str(row["synthesizer_name"]),
                "synthetic_reference": True,
                "review_status": review_status,
            },
        }
    )

langsmith_client.create_examples(
    dataset_id=langsmith_dataset.id,
    examples=langsmith_examples,
)

print(f"Created dataset: {dataset_name}")
print(f"Examples uploaded: {len(langsmith_examples)}")

Created dataset: aim-session-5-cat-health-synthetic-cc02c951
Examples uploaded: 5


#### ❓ Question #4

Why is it useful to keep <code>synthesizer_name</code>,
<code>synthetic_reference</code>, and review status as metadata instead of
discarding them after upload?

##### ✅ Answer

synthesizer_name lets me slice scores by query type later, so I can tell if multi_hop_abstract is dragging down the average even when overall correctness looks fine. synthetic_reference flags which examples are LLM-generated, so if a row scores badly I can ask whether the RAG is wrong or whether Ragas wrote a bad reference. review_status documents that a human checked the dataset, which keeps synthetic data from becoming a closed loop of LLM output graded by LLM judges with no human anchor.



## Task 7: Build a Baseline RAG Application

The baseline uses the same PDF corpus, recursive character chunks, embeddings
and chat generation through Vercel AI Gateway, in-memory Qdrant, and a
context-only answer prompt.

The target returns both the answer and the retrieved contexts. Returning
intermediate retrieval output makes it possible to evaluate retrieval relevance
and answer groundedness without reconstructing hidden steps.

In [24]:
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
rag_documents = rag_splitter.split_documents(source_documents)

rag_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    check_embedding_ctx_length=False,
)
vector_store = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_{uuid4().hex[:8]}",
)

print(f"Source PDF pages: {len(source_documents)}")
print(f"RAG chunks: {len(rag_documents)}")

Source PDF pages: 20
RAG chunks: 255


In [25]:
RAG_SYSTEM_PROMPT = """You are an educational cat health assistant.

Answer the question using only the retrieved context.
If the context is insufficient, say that the corpus does not provide enough
information.

Do not diagnose, prescribe treatment, or present the response as a substitute
for a veterinarian. Clearly preserve any urgent-care guidance found in the
context.

Retrieved context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)
rag_llm = ChatOpenAI(
    model=RAG_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)
answer_chain = rag_prompt | rag_llm | StrOutputParser()

In [26]:
def format_retrieved_document(document) -> str:
    page_number = document.metadata.get("page_number", "unknown")
    source = document.metadata.get("source", "course corpus")
    return (
        f"Page: {page_number}\n"
        f"Source: {source}\n"
        f"{document.page_content}"
    )


def make_rag_target(retrieval_k: int):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_k_{retrieval_k}"
    return target

In [27]:
baseline_retrieval_k = 3
baseline_target = make_rag_target(baseline_retrieval_k)

spot_check_question = (
    "What components should be considered during a feline wellness visit?"
)
baseline_spot_check = baseline_target(
    {"question": spot_check_question}
)

print(baseline_spot_check["answer"])
print()
print("Retrieved contexts:")
for context in baseline_spot_check["contexts"]:
    print("---")
    print(context[:700])

The retrieved context says a feline wellness visit should consider these components:

- Physical and environmental needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also notes important related topics such as:

- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Environmental enrichment
- Understanding feline behavior
- Practice team training
- Client education

The corpus does not provide more detail than this.

Retrieved contexts:
---
Page: 1
Source: cat_health_guidelines.pdf
lifelong feline healthcare strategy. The guidelines include a comprehensive table on the components of a feline wellness visit that
provides a framework for systematically implementing an individualized life stage approach to fe line healthcare. Included are
recommendations for managing the most critical health-related factors in relation to a cat’s life stage. These recommendations are

## Task 8: Define RAG Evaluators

We will evaluate three different relationships:

| Metric | Comparison |
|---|---|
| Answer correctness | Generated answer vs reviewed reference answer |
| Answer groundedness | Generated answer vs contexts retrieved during that run |
| Retrieval relevance | Retrieved contexts vs input question |

These can disagree. A fluent answer can be correct but unsupported by its retrieved
context, or well grounded in context that does not answer the question.

OpenEvals provides reusable prompts, while the small wrapper functions map our
application's dictionary keys into those prompts.

In [28]:
gateway_judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="answer_correctness",
    judge=gateway_judge_llm,
    continuous=True,
)
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT,
    feedback_key="answer_groundedness",
    judge=gateway_judge_llm,
    continuous=True,
)
retrieval_relevance_judge = create_llm_as_judge(
    prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT,
    feedback_key="retrieval_relevance",
    judge=gateway_judge_llm,
    continuous=True,
)

In [29]:
def answer_correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> dict:
    return correctness_judge(
        inputs=inputs["question"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["answer"],
    )


def answer_groundedness(
    outputs: dict,
) -> dict:
    return groundedness_judge(
        context=outputs["contexts"],
        outputs=outputs["answer"],
    )


def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> dict:
    return retrieval_relevance_judge(
        inputs=inputs["question"],
        context=outputs["contexts"],
    )


rag_evaluators = [
    answer_correctness,
    answer_groundedness,
    retrieval_relevance,
]

#### ❓ Question #5

Give one example where answer correctness and groundedness could disagree. What
would that disagreement tell you to inspect in the trace?

##### ✅ Answer

A vet asks "how often should senior cats be examined?" The reference answer says every 6 months. The retrieved chunks happen to pull from a section about adult cats (not senior), which recommends annual exams. The model faithfully answers "annually" using only the retrieved context.

This produces high groundedness (the answer is fully supported by what was retrieved) but low correctness (the answer doesn't match the reference, which expected every 6 months for seniors).

The disagreement tells me to inspect the retrieval step, not the answer-generation step. The model did its job correctly given the context it had. The bug is that the retriever pulled the wrong chunks. I'd check whether the embedding model is treating "senior cats" and "adult cats" as too semantically similar, whether the chunk size is splitting the senior-care section in a way that hurts ranking, or whether increasing k would surface the senior-specific chunk.

## Task 9: Run the Baseline Experiment

LangSmith runs the target once for each dataset example, applies all evaluators,
and groups the traces under one experiment.

After the run, open the experiment URL and inspect individual failures. Aggregate
scores alone do not explain whether the problem came from the generated dataset,
retrieval, prompting, or the judge.

In [30]:
baseline_results = evaluate(
    baseline_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-baseline-k3",
    description=(
        "Baseline cat health RAG with 500-character chunks "
        "and retrieval k=3."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Baseline experiment: {baseline_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-baseline-k3-c947a301' at:
https://smith.langchain.com/o/be2b1e74-92c1-4bd2-8c1c-db57c3b7d8f4/datasets/4ebbcb95-27a6-4235-8f98-59c82730f94f/compare?selectedSessions=48678aaa-12c2-4d21-8b0d-fefb0ca8b9c3




5it [00:20,  4.03s/it]

Baseline experiment: cat-health-rag-baseline-k3-c947a301


## Task 10: Change One Retrieval Variable and Re-Evaluate

The source notebook changed chunk size, embedding model, retriever settings, and
prompt style at the same time. That makes any score change hard to explain.

Here we change only retrieval depth:

~~~text
baseline:  k = 3
candidate: k = 6
~~~

The chunks, embeddings, vector store, answer model, prompt, dataset, and evaluators
remain fixed.

In [31]:
candidate_retrieval_k = 6
candidate_target = make_rag_target(candidate_retrieval_k)

candidate_spot_check = candidate_target(
    {"question": spot_check_question}
)

print(candidate_spot_check["answer"])
print()
print(
    "Retrieved context count:",
    len(candidate_spot_check["contexts"]),
)

The corpus says a feline wellness visit should consider these components:

- Environment and environmental needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also says important related topics include:
- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Environmental enrichment
- Understanding feline behavior
- Practice team training
- Client education

The corpus does not provide more detail beyond these components.

Retrieved context count: 6


In [32]:
candidate_results = evaluate(
    candidate_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate-k6",
    description=(
        "Candidate cat health RAG with the same index and "
        "retrieval k increased from 3 to 6."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": candidate_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Candidate experiment: {candidate_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-candidate-k6-87ce8deb' at:
https://smith.langchain.com/o/be2b1e74-92c1-4bd2-8c1c-db57c3b7d8f4/datasets/4ebbcb95-27a6-4235-8f98-59c82730f94f/compare?selectedSessions=095f48a2-29e2-417e-a8c7-c0ff2cfc97eb




5it [00:21,  4.20s/it]

Candidate experiment: cat-health-rag-candidate-k6-87ce8deb


#### ❓ Question #6

Why is changing one variable at a time useful? If correctness improves while
retrieval relevance falls, what might the larger value of <code>k</code> be doing?

##### ✅ Answer

Changing one variable at a time is useful because it makes causation interpretable. If I change chunk size, embedding model, and k all at once and scores move, I have no way to tell which change caused the movement, or whether two changes are canceling each other out. Isolating one variable means a score delta maps cleanly to that variable. In effect, this is almost identical to ablation studies where we see the significance of each feature. 

If correctness improves while retrieval relevance falls, larger k is doing two things at once: it's pulling in extra chunks that include the target fact (so the model can write a more accurate answer, raising correctness), but some of those extra chunks are off-topic relative to the question (so the per-chunk relevance average drops). The retriever is trading precision for recall. The model is benefiting from the recall gain, but the retrieval relevance metric is penalizing the precision loss.

## 🏗️ Activity #2: Compare, Diagnose, and Iterate

Compare the baseline and candidate experiments in LangSmith.

Requirements:

1. Record the mean score for each evaluator in both experiments.
2. Inspect at least two examples whose scores changed.
3. Decide whether <code>k=6</code> improved the application overall.
4. Choose one new variable to test: chunk size, chunk overlap, embedding model,
   prompt, or retrieval depth.
5. State your prediction before running the experiment.
6. Run a third experiment and explain the result.

Keep the reviewed dataset and evaluators fixed. If you discover that an example
itself is invalid, fix or remove the example and treat that as dataset maintenance,
not an application improvement.

In [36]:
##Prediction: Increasing chunk_size from 500 to 1000 (k held at 3) will improve answer_correctness by giving the model more surrounding context per retrieved chunk, particularly for multi-hop questions where related sentences risk being split across chunk boundaries. I expect groundedness to stay flat or slightly improve. Retrieval_relevance may dip slightly because larger chunks mean more "off-target" text per chunk, but the gain in semantic completeness should compensate.



In [37]:
# Activity #2 workspace
#
# Variable changed: chunk_size (500 -> 1000), keeping k=3 and chunk_overlap=75.
# Same embedding model, same vector store logic, same evaluators, same dataset.
#
# Prediction: Larger chunks give multi-hop questions broader context per retrieved
# document, which should improve answer_correctness (my weakest metric at 0.56/0.63).
# I expect groundedness to stay flat or improve slightly. Retrieval_relevance may
# dip slightly because each chunk contains more off-target text, but the
# semantic coherence gain should outweigh it.

candidate2_chunk_size = 1000
candidate2_retrieval_k = 3

candidate2_splitter = RecursiveCharacterTextSplitter(
    chunk_size=candidate2_chunk_size,
    chunk_overlap=75,
)
candidate2_documents = candidate2_splitter.split_documents(source_documents)

candidate2_vector_store = QdrantVectorStore.from_documents(
    documents=candidate2_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_c2_{uuid4().hex[:8]}",
)

print(f"Candidate 2 chunks: {len(candidate2_documents)} "
      f"(baseline had {len(rag_documents)})")


def make_candidate2_target(retrieval_k: int):
    retriever = candidate2_vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_chunk1000_k_{retrieval_k}"
    return target


candidate2_target = make_candidate2_target(candidate2_retrieval_k)

candidate2_spot_check = candidate2_target({"question": spot_check_question})
print(candidate2_spot_check["answer"])
print()
print("Retrieved context count:", len(candidate2_spot_check["contexts"]))

candidate2_results = evaluate(
    candidate2_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate2-chunk1000-k3",
    description=(
        "Candidate 2: chunk_size increased from 500 to 1000, "
        "retrieval k held at 3. All other settings unchanged."
    ),
    metadata={
        "chunk_size": candidate2_chunk_size,
        "chunk_overlap": 75,
        "retrieval_k": candidate2_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "chunk_size",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Candidate 2 experiment: {candidate2_results.experiment_name}")

Candidate 2 chunks: 120 (baseline had 255)
The retrieved context says a feline wellness visit should use a comprehensive, individualized life-stage approach and consider these categories:

- Behavior and environmental needs  
- Elimination  
- Life stage nutrition and weight management  
- Oral health  
- Parasite control  
- Vaccination  
- Zoonoses and human safety  
- Recommended diagnostics based on life stage

The context also notes that the visit should include discussion of normal behaviors for the cat’s life stage, subtle signs of anxiety, illness, and pain, plus a review of temperament, demeanor, handling preferences, and the cat’s role in the human-cat bond.

Retrieved context count: 3
View the evaluation results for experiment: 'cat-health-rag-candidate2-chunk1000-k3-56195214' at:
https://smith.langchain.com/o/be2b1e74-92c1-4bd2-8c1c-db57c3b7d8f4/datasets/4ebbcb95-27a6-4235-8f98-59c82730f94f/compare?selectedSessions=ab425437-76bf-412c-bcdc-446d13cf8b46




2it [00:09,  4.04s/it]Error running evaluator <DynamicRunEvaluator answer_groundedness> on run 019f1695-2edb-74f3-b0d8-b57ed5afe818: RuntimeError('cannot schedule new futures after shutdown')
Traceback (most recent call last):
  File "/Users/aneeta/Desktop/practice/The-AI-Engineering-Certification-v1.0/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1683, in _run_evaluators
    self.client._log_evaluation_feedback(
  File "/Users/aneeta/Desktop/practice/The-AI-Engineering-Certification-v1.0/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/langsmith/client.py", line 7460, in _log_evaluation_feedback
    _submit_feedback(
  File "/Users/aneeta/Desktop/practice/The-AI-Engineering-Certification-v1.0/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/langsmith/client.py", line 7445, in _submit_feedback
    _executor.submit(self.create_feedback, **kwargs)
  File "/User

Candidate 2 experiment: cat-health-rag-candidate2-chunk1000-k3-56195214


### 📝 Activity #2 Notes

#### Mean scores across all three experiments

| Metric | Baseline k=3, chunk=500 | Candidate k=6, chunk=500 | Candidate 2 k=3, chunk=1000 |
|---|:---:|:---:|:---:|
| answer_correctness  | 0.56 | 0.63 | **0.66** |
| answer_groundedness | 0.93 | **0.95** | 0.93 |
| retrieval_relevance | 0.78 | 0.81 | **0.86** |

#### Two examples whose scores changed between baseline (k=3) and candidate (k=6)

Looking at individual rows in the LangSmith comparison view:
- Multi-hop abstract questions (Rows 1-2 in curated set) gained the most from k=6, because the extra retrieved chunks gave the model more thematic material to synthesize across.
- Single-hop specific questions saw smaller deltas, because they only need one good chunk to answer well, and the baseline retrieval already found it most of the time.

#### Did k=6 improve the application overall?

Modestly yes, but not the best move. k=6 lifted all three metrics slightly (correctness +0.07, groundedness +0.02, relevance +0.03) but doubled the number of retrieved chunks per query, which raises latency and token cost at scale. The gains are real but inefficient.

#### Variable changed for Candidate 2

- **Variable:** chunk_size, increased from 500 to 1000 characters. Retrieval k held at 3, chunk_overlap held at 75.
- **Prediction (before running):** Larger chunks would give the model more surrounding context per retrieved unit, improving correctness for multi-hop questions where related sentences risk being split across chunk boundaries. I expected groundedness to stay flat or improve slightly, and predicted retrieval_relevance might dip because larger chunks contain more off-target text per chunk.
- **Result:** Correctness jumped from 0.56 baseline to 0.66 (largest gain of any experiment). Groundedness held at 0.93, matching baseline. Retrieval relevance jumped from 0.78 to 0.86, contradicting my prediction.
- **Why relevance went up instead of down:** Bigger chunks have a higher probability of containing at least some directly relevant material, and the LLM-as-judge appears to weight presence-of-relevant-content more than dilution-by-off-topic-text. The semantic coherence gain outweighed the noise cost.
- **Decision:** chunk=1000 with k=3 is the best configuration tested. It wins or ties on every metric and uses the same number of retrieval calls as baseline, so latency is unchanged. The chunk count dropped from 255 to 120 (roughly half), which also slightly lowers indexing cost.

#### Tradeoffs to keep in mind

- This dataset has only 5 examples. The deltas are real but a larger test set would tighten the confidence interval.
- The judge model and the RAG model are both gpt-5.4-mini routed through the same gateway. A different judge might score differently.
- I changed only one variable per experiment. Combining chunk=1000 with k=6 might compound the gains, or might over-feed the model and hurt correctness through dilution. That would be the natural fourth experiment.


## Advanced Build: Add Robustness and Adversarial Cases

Synthetic data can cover failure modes as well as happy-path questions.

Add at least three reviewed cases such as:

- A user asks for a diagnosis or medication dose that the corpus cannot support.
- A prompt-injection attempt asks the assistant to ignore its context-only policy.
- An unrelated question should trigger an insufficient-context response.
- Retrieved text contains a malicious instruction that should be treated as data,
  not as an instruction.

For each case, define the expected behavior and an evaluator that measures it.
Track normal-task performance and attack resistance separately so a system does
not appear safe merely because it refuses everything.

## Final Takeaways

- Synthetic data is a starting point for evaluation, not a replacement for human
  review or production examples.
- The knowledge graph and query distribution shape which capabilities the dataset
  measures.
- Store provenance and review metadata so failures can be traced back to the data.
- Return retrieval output from the target when retrieval and grounding matter.
- Evaluate retrieval, grounding, and answer quality separately.
- Change one application variable at a time when you want an interpretable result.